In [ ]:
///- --package .
open sm'_operators

#### site (test)

In [ ]:
///- --test static

open testing

In [1]:
type file = { path : string; content : string }

inl layout (title : string) (body : string) : string =
    "<!doctype html><html lang=\"en\"><head><meta charset=\"utf-8\">"
    ++# "<meta name=\"viewport\" content=\"width=device-width, initial-scale=1\">"
    ++# "<title>" ++# html.escape title ++# "</title>"
    ++# "<link rel=\"stylesheet\" href=\"/style.css\"></head>"
    ++# "<body><header><a class=\"home\" href=\"/\">Spiral Blog</a></header><main>"
    ++# body ++# "</main></body></html>"

inl entry_file (e : page.entry) : file =
    { path = e.slug ++# "/index.html"; content = layout e.title (page.render_entry e) }

inl rec entry_files (entries : list page.entry) : list file =
    match entries with
    | Nil => Nil
    | Cons (e, rest) => Cons (entry_file e, entry_files rest)

inl base_url () : string = "https://i574n.github.io/blog"

inl files (entries : list page.entry) : list file =
    Cons (
        { path = "index.html"; content = layout "Spiral Blog" (index.render_index entries) },
        Cons (
            { path = "style.css"; content = style.css () },
            Cons (
                { path = "rss.xml"; content = rss.render_feed "Spiral Blog" (base_url ()) entries },
                Cons (
                    {
                        path = "articles/index.html"
                        content = layout "Articles - Spiral Blog" (index.render_index (index.only page.Article entries))
                    },
                    Cons (
                        {
                            path = "releases/index.html"
                            content =
                                layout "Releases - Spiral Blog" (index.render_index (index.only page.Release entries))
                        },
                        entry_files entries
                    )
                )
            )
        )
    )

inl rec paths (fs : list file) : string =
    match fs with
    | Nil => ""
    | Cons (f, Nil) => f.path
    | Cons (f, rest) => f.path ++# " " ++# paths rest

#### the_site_has_an_index_and_one_folder_per_post (test)

In [2]:
///- --test

files (Cons (page.sample (), Nil))
|> paths
|> _assert_eq "index.html style.css rss.xml articles/index.html releases/index.html sample/index.html"

__assert_eq { expected = index.html style.css rss.xml articles/index.html releases/index.html sample/index.html }

#### every_page_shares_the_layout (test)

In [3]:
///- --test

inl rec all_wrapped (fs : list file) : bool =
    match fs with
    | Nil => true
    | Cons (f, rest) when f.path |> sm'.ends_with ".html" =>
        (f.content |> sm'.starts_with "<!doctype html>")
        && (f.content |> sm'.ends_with "</main></body></html>")
        && all_wrapped rest
    | Cons (_, rest) => all_wrapped rest
files (Cons (page.sample (), Nil)) |> all_wrapped |> _assert_eq true

__assert_eq { expected = true }

#### the_post_page_title_is_escaped_into_the_head (test)

In [4]:
///- --test

inl e = page.sample ()
files (Cons ({ e with title = "A & B" }, Nil))
|> fun fs =>
    match fs with
    | Cons (_, Cons (_, Cons (_, Cons (_, Cons (_, Cons (post_file, _)))))) =>
        post_file.content |> sm'.contains "<title>A &amp; B</title>" |> _assert_eq true
    | _ => failwith "expected a post file"

__assert_eq { expected = true }